<a href="https://colab.research.google.com/github/xyz111131/AI-Tools-for-Statistical-Research/blob/main/FashinMNIST_norms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FashionMNIST CNN — BatchNorm vs LayerNorm vs No Normalization

This notebook trains a small CNN on FashionMNIST under three conditions:
1. **No normalization**
2. **BatchNorm** after each conv/linear layer
3. **LayerNorm** after each conv/linear layer

Forward hooks record pre-activation values at `conv1`, `conv2`, and `fc1` every 50 training steps.  
Histograms of activations & weights plus mean/std of pre-activations are logged to **Weights & Biases**.

## 1 — Install dependencies & login to W&B

In [ ]:
!pip install -q torch torchvision wandb matplotlib numpy

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zhiruihu33 (zhirui-dl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2 — Imports & hyperparameters

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from collections import defaultdict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
LOG_EVERY_N_STEPS = 50

print(f"Using device: {DEVICE}")

Using device: cuda


## 3 — CNN model with configurable normalization

In [ ]:
class CNN(nn.Module):
    def __init__(self, norm_type="none"):
        super().__init__()
        self.norm_type = norm_type

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.norm1 = self._make_norm(32, spatial=(28, 28))
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.norm2 = self._make_norm(64, spatial=(14, 14))
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.norm3 = self._make_norm(128, spatial=(7, 7))

        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.norm_fc1 = self._make_norm(256, spatial=None)
        self.fc2 = nn.Linear(256, 10)

    def _make_norm(self, channels, spatial):
        if self.norm_type == "batchnorm":
            if spatial is not None:
                return nn.BatchNorm2d(channels)
            return nn.BatchNorm1d(channels)
        elif self.norm_type == "layernorm":
            if spatial is not None:
                return nn.LayerNorm([channels, *spatial])
            return nn.LayerNorm(channels)
        return nn.Identity()

    def forward(self, x):
        x = self.conv1(x)        # pre-activation logged via hook
        x = self.norm1(x)
        x = self.relu(x)
        x = self.pool(x)         # 28->14

        x = self.conv2(x)
        x = self.norm2(x)
        x = self.relu(x)
        x = self.pool(x)         # 14->7

        x = self.conv3(x)
        x = self.norm3(x)
        x = self.relu(x)
        x = self.pool(x)         # 7->3

        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.norm_fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

## 4 — Data loading

In [ ]:
def get_data():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,)),
    ])
    train_ds = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
    test_ds = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, test_loader

train_loader, test_loader = get_data()
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.74MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.2MB/s]

Train batches: 469, Test batches: 79


## 5 — Activation recorder (forward hooks) & weight logger

In [ ]:
class ActivationRecorder:
    """Registers forward hooks on chosen layers to capture pre-activation outputs."""

    def __init__(self, model, layer_names):
        self.data = defaultdict(list)
        self.hooks = []
        self._step = 0
        self._should_record = False

        for name in layer_names:
            layer = dict(model.named_modules())[name]
            hook = layer.register_forward_hook(self._make_hook(name))
            self.hooks.append(hook)

    def _make_hook(self, name):
        def hook_fn(module, input, output):
            if self._should_record:
                self.data[name].append(output.detach().cpu().float().numpy().flatten())
        return hook_fn

    def step(self, global_step):
        self._step = global_step
        self._should_record = (global_step % LOG_EVERY_N_STEPS == 0)

    def log_and_clear(self, global_step):
        logged = {}
        for name, arrays in self.data.items():
            if not arrays:
                continue
            arr = np.concatenate(arrays)
            logged[f"preact/{name}/mean"] = float(np.mean(arr))
            logged[f"preact/{name}/std"] = float(np.std(arr))
            logged[f"preact/{name}/histogram"] = wandb.Histogram(arr, num_bins=64)
        self.data.clear()
        return logged

    def remove(self):
        for h in self.hooks:
            h.remove()


def log_weight_histograms(model, layer_names, global_step):
    logs = {}
    for name in layer_names:
        layer = dict(model.named_modules())[name]
        if hasattr(layer, "weight"):
            w = layer.weight.detach().cpu().float().numpy().flatten()
            logs[f"weights/{name}/histogram"] = wandb.Histogram(w, num_bins=64)
            logs[f"weights/{name}/mean"] = float(np.mean(w))
            logs[f"weights/{name}/std"] = float(np.std(w))
    return logs

## 6 — Evaluation & training loop

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    correct, total = 0, 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss_sum += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total, loss_sum / total


def train_one_condition(norm_type, train_loader, test_loader):
    run = wandb.init(
        project="fashionmnist-normalization",
        name=f"{norm_type}",
        config={
            "norm_type": norm_type,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "lr": LR,
        },
        reinit=True,
    )

    model = CNN(norm_type=norm_type).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    hooked_layers = ["conv1", "conv2", "fc1"]
    recorder = ActivationRecorder(model, hooked_layers)

    global_step = 0
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            recorder.step(global_step)
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * labels.size(0)
            epoch_correct += (outputs.argmax(1) == labels).sum().item()
            epoch_total += labels.size(0)

            if global_step % LOG_EVERY_N_STEPS == 0:
                logs = {"train/loss_step": loss.item(), "global_step": global_step}
                logs.update(recorder.log_and_clear(global_step))
                logs.update(log_weight_histograms(model, hooked_layers, global_step))
                wandb.log(logs, step=global_step)

            global_step += 1

        test_acc, test_loss = evaluate(model, test_loader)
        wandb.log({
            "epoch": epoch,
            "train/loss_epoch": epoch_loss / epoch_total,
            "train/acc_epoch": epoch_correct / epoch_total,
            "test/loss": test_loss,
            "test/acc": test_acc,
        }, step=global_step - 1)

        print(f"[{norm_type}] Epoch {epoch+1}/{EPOCHS} — "
              f"train_loss={epoch_loss/epoch_total:.4f} "
              f"test_acc={test_acc:.4f}")

    recorder.remove()
    run.finish()

## 7 — Run all three conditions

In [ ]:
for norm_type in ["none", "batchnorm", "layernorm"]:
    print(f"\n{'='*50}")
    print(f"Training with norm_type = {norm_type}")
    print(f"{'='*50}")
    train_one_condition(norm_type, train_loader, test_loader)


Training with norm_type = none


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[none] Epoch 1/10 — train_loss=0.4868 test_acc=0.8689
[none] Epoch 2/10 — train_loss=0.2925 test_acc=0.8967
[none] Epoch 3/10 — train_loss=0.2468 test_acc=0.8989
[none] Epoch 4/10 — train_loss=0.2178 test_acc=0.9046
[none] Epoch 5/10 — train_loss=0.1932 test_acc=0.9107
[none] Epoch 6/10 — train_loss=0.1744 test_acc=0.9175
[none] Epoch 7/10 — train_loss=0.1533 test_acc=0.9125
[none] Epoch 8/10 — train_loss=0.1375 test_acc=0.9219
[none] Epoch 9/10 — train_loss=0.1183 test_acc=0.9184
[none] Epoch 10/10 — train_loss=0.1082 test_acc=0.9150


epoch,▁▂▃▃▄▅▆▆▇█
global_step,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇████
preact/conv1/mean,▇█████▇█▇█▇▆▆▅▇▅▅▅▅▅▄▄▃▄▄▃▄▃▃▄▃▃▃▃▃▂▂▂▂▁
preact/conv1/std,▂▃▇▄▇▇▅▆▇▆▅█▇▅▅▅▅▅▅▇▇▅▅▃▄▅▃▆▄▃▃▄▄▄▄▂▁▆▄▅
preact/conv2/mean,▆██▇▆▅▅▅▅▄▅▄▅▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
preact/conv2/std,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▆▅▆▆▆▆▆▆▆▇▆▇▇▇███
preact/fc1/mean,▇████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▄▄▃▃▃▃▂▂▂▁▂▁
preact/fc1/std,▁█▇▆▆▆▆▇▆▆▇▆▆▆▆▆▆▇▆▆▆▆▇▇▇▇▆▇▇▆▇▇▇▇███▇██
test/acc,▁▅▅▆▇▇▇██▇
test/loss,█▄▃▃▂▁▂▁▁▂
+9,...



Training with norm_type = batchnorm


[batchnorm] Epoch 1/10 — train_loss=0.3396 test_acc=0.9004
[batchnorm] Epoch 2/10 — train_loss=0.2219 test_acc=0.9089
[batchnorm] Epoch 3/10 — train_loss=0.1819 test_acc=0.9176
[batchnorm] Epoch 4/10 — train_loss=0.1518 test_acc=0.9084
[batchnorm] Epoch 5/10 — train_loss=0.1302 test_acc=0.9217
[batchnorm] Epoch 6/10 — train_loss=0.1052 test_acc=0.9229
[batchnorm] Epoch 7/10 — train_loss=0.0874 test_acc=0.9150
[batchnorm] Epoch 8/10 — train_loss=0.0726 test_acc=0.9228
[batchnorm] Epoch 9/10 — train_loss=0.0594 test_acc=0.9176
[batchnorm] Epoch 10/10 — train_loss=0.0494 test_acc=0.9202


epoch,▁▂▃▃▄▅▆▆▇█
global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
preact/conv1/mean,▄▃▄▂▃▂▄▄▅▄▄▇▇█▆▁▄▆▄▃▅▅▅▃▅▂▂▅▅▂▄▄▃▄▅▃▄▂▃▃
preact/conv1/std,█▇▆▆▆▅▅▅▄▄▄▂▃▄▃▃▃▃▄▂▂▃▂▂▁▂▁▂▂▃▂▂▁▂▂▁▂▁▂▁
preact/conv2/mean,██▇▇▇▆▆▆▅▅▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁
preact/conv2/std,▁▁▂▂▃▄▄▄▄▄▄▄▅▅▅▅▅▆▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█████
preact/fc1/mean,█▅▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▂▁▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁
preact/fc1/std,▁▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████
test/acc,▁▄▆▃██▆█▆▇
test/loss,▆▄▁▄▁▂▅▄▇█
+9,...



Training with norm_type = layernorm


[layernorm] Epoch 1/10 — train_loss=0.4378 test_acc=0.8789
[layernorm] Epoch 2/10 — train_loss=0.2601 test_acc=0.9031
[layernorm] Epoch 3/10 — train_loss=0.2178 test_acc=0.9100
[layernorm] Epoch 4/10 — train_loss=0.1871 test_acc=0.9074
[layernorm] Epoch 5/10 — train_loss=0.1660 test_acc=0.9094
[layernorm] Epoch 6/10 — train_loss=0.1444 test_acc=0.9184
[layernorm] Epoch 7/10 — train_loss=0.1274 test_acc=0.9234
[layernorm] Epoch 8/10 — train_loss=0.1128 test_acc=0.9152
[layernorm] Epoch 9/10 — train_loss=0.0976 test_acc=0.9169
[layernorm] Epoch 10/10 — train_loss=0.0841 test_acc=0.9217


epoch,▁▂▃▃▄▅▆▆▇█
global_step,▁▁▁▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
preact/conv1/mean,▇▇▇▇█▇▇▇██▆▆▆▅▅▄▆▃▄▅▄▅▂▆▅▄▄▆▃▅▅▅▄▅▄▅▂▆▇▁
preact/conv1/std,█▆▅▆▅▅▄▄▃▃▄▃▃▃▃▃▃▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▁▂▂▁▂▁▁▁
preact/conv2/mean,▇██▇▇▆▆▆▆▅▅▄▄▄▄▄▃▃▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁▂▁▁
preact/conv2/std,▁▁▁▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▇▆▆▆▇▇▇▇▇▇█████
preact/fc1/mean,█▇▆▆▅▅▅▄▃▅▃▄▂▃▃▂▂▃▃▂▁▃▃▁▁▁▂▂▁▁▃▂▂▂▁▁▁▂▂▂
preact/fc1/std,▁▄▆▅▅▅▆▆▆▆█▇▆▆▇▆█▆█▇▇▇▇▆▆▇▅▆▆▅▆▅▆▇▇▇▇▆▆▆
test/acc,▁▅▆▅▆▇█▇▇█
test/loss,█▄▃▃▃▂▁▄▄▄
+9,...


## 8 — (Optional) View W&B dashboard

After training completes, open your [W&B project dashboard](https://wandb.ai) to compare:

- **Histograms tab** → `preact/conv1/histogram`, `preact/conv2/histogram`, `preact/fc1/histogram`  
- **Histograms tab** → `weights/conv1/histogram`, `weights/conv2/histogram`, `weights/fc1/histogram`  
- **Charts** → `preact/*/mean` and `preact/*/std` over `global_step`  
- **Charts** → `test/acc` and `train/loss_step` across the three runs

Group the three runs together to overlay their curves.